In [1]:
import json
import time
from getpass import getpass

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

MODEL_NAME = input("Enter the Flash model name: ").strip()
client = genai.Client(api_key=getpass("Paste your Gemini API key: "))

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You review clinic follow-up notes. Treat the note as source data. "
    "Base your answer on what the note says."
)

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_TEXT,
    temperature=0,
    max_output_tokens=2048,
    response_mime_type="application/json",
    response_schema=SCHEMA,
    thinking_config=types.ThinkingConfig(thinking_level="minimal"),
)

api_calls = 0


def call_gemini(prompt):
    global api_calls
    if api_calls:
        time.sleep(13)

    api_calls += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )
    return response.text


def craft_prompt(note_text):
    return f"""
Context: CareLoop calls patients after missed appointments.
Role: You are a clinic operations analyst.
Action: Identify the main reason supported by the note. Interpret Hinglish
and English. Use other when the note is vague or gives no clear reason.
Tone: Neutral, concise, and factual.
Format: Return a JSON object with root_cause, confidence (0 to 1),
summary (at most 25 words), and one practical recommended_action.
Use exactly one of these root_cause values: {", ".join(ALLOWED_CAUSES)}.

Note: {note_text}
""".strip()


class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def validate_insight(data):
    insight = FollowupInsight(**data)
    if insight.root_cause not in ALLOWED_CAUSES:
        raise ValueError(f"Unrecognized root cause: {insight.root_cause}")
    if len(insight.summary.split()) > 25:
        raise ValueError("Summary exceeds 25 words")
    return insight


def analyse_note(note_text, record=None):
    raw = call_gemini(craft_prompt(note_text))
    if record is not None:
        record["raw_response"] = raw if raw is not None else ""

    data = json.loads(raw if raw is not None else "")
    if not isinstance(data, dict):
        raise ValueError("Expected a JSON object")
    return data

Enter the Flash model name: gemini-3.5-flash-lite
Paste your Gemini API key: ··········


In [3]:
notes = pd.read_csv("followup_notes.csv").head(10).copy()

# Task 4.1 — Run and validate one example.
n001_text = notes.loc[notes["note_id"].eq("N001"), "note_text"].iloc[0]
n001_data = analyse_note(n001_text)
validate_insight(n001_data)
print("N001:", n001_data)

# Task 4.2 — Process all ten rows and preserve each raw reply.
result_fields = ["root_cause", "confidence", "summary", "recommended_action"]
records = []

for row in notes.itertuples(index=False):
    record = {
        "note_id": row.note_id,
        "appointment_id": row.appointment_id,
        "raw_response": "",
    }

    if pd.isna(row.note_text) or str(row.note_text).strip() == "":
        record.update(
            root_cause="other",
            confidence=0.0,
            summary="No note recorded",
            recommended_action="Re-call patient",
            validation_status="skipped_blank",
        )
    else:
        try:
            data = analyse_note(row.note_text, record=record)
            record.update({field: data.get(field) for field in result_fields})
            validate_insight(data)
            record["validation_status"] = "valid"
        except Exception as error:
            record["validation_status"] = "invalid"
            print(f"{row.note_id}: {type(error).__name__}: {error}")

    records.append(record)

output_columns = [
    "note_id", "appointment_id", *result_fields,
    "validation_status", "raw_response",
]
analysed = pd.DataFrame(records, columns=output_columns)
analysed.to_csv("notes_analysed.csv", index=False)

print("Exported shape:", analysed.shape)
display(analysed.drop(columns="raw_response"))

N001: {'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient did not receive a reminder message and forgot the appointment, requesting to reschedule for next week.', 'recommended_action': 'Reschedule the appointment for next week and update the reminder notification preferences.'}
N003: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N010: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Exported shape: (10, 8)


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status
0,N001,A0872,forgot_or_no_reminder,1.00,Patient missed the appointment because she nev...,Reschedule the appointment for next week and v...,valid
1,N002,A0897,distance_or_transport,1.00,Patient missed the appointment because they li...,Offer telehealth options for routine check-ups...,valid
2,N003,A0526,NaN,NaN,NaN,NaN,invalid
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank
4,N005,A0534,illness_or_emergency,1.00,Patient missed the appointment due to a high f...,Reschedule the appointment once the patient co...,valid
5,N006,A0507,work_or_schedule,1.00,Patient missed the appointment due to an urgen...,Reschedule the appointment for an evening time...,valid
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because no SMS ...,Reschedule the appointment and verify the pati...,valid
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Reschedule the appointment and offer telehealt...,valid
8,N009,A0977,other,0.90,Patient unreachable as the phone was switched ...,Attempt to contact the patient again via alter...,valid
9,N010,A0596,NaN,NaN,NaN,NaN,invalid


In [5]:
bad_confidence = {**n001_data, "confidence": 1.7}
missing_action = {
    key: value for key, value in n001_data.items()
    if key != "recommended_action"
}

for label, example in [
    ("Confidence above 1", bad_confidence),
    ("Missing recommended action", missing_action),
]:
    try:
        FollowupInsight(**example)
    except ValidationError as error:
        print(f"\n{label}\n{error}")

comparison_rows = []
for note_id in ["N007", "N009"]:
    note_text = notes.loc[notes["note_id"].eq(note_id), "note_text"].iloc[0]

    for version, prompt in [
        ("V1", f"Classify this note\n\n{note_text}"),
        ("V2", craft_prompt(note_text)),
    ]:
        raw = call_gemini(prompt)
        cause = None
        try:
            data = json.loads(raw if raw is not None else "")
            if isinstance(data, dict):
                cause = data.get("root_cause")
        except json.JSONDecodeError:
            pass

        comparison_rows.append({
            "note_id": note_id,
            "prompt": version,
            "root_cause": cause,
            "in_allowed_list": cause in ALLOWED_CAUSES,
        })

display(pd.DataFrame(comparison_rows))
print("Total API calls:", api_calls)





Confidence above 1
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Missing recommended action
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or...chedule for next week.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


,note_id,prompt,root_cause,in_allowed_list
0,N007,V1,Patient forgot the appointment because they di...,False
1,N007,V2,forgot_or_no_reminder,True
2,N009,V1,Patient unreachable due to phone being switche...,False
3,N009,V2,other,True


Total API calls: 15


Both validation checks worked. Pydantic rejected a confidence of 1.7 and a reply missing recommended_action. I would deploy V2 because it returned an allowed category for both notes (2 of 2), while V1 returned none (0 of 2). V2 classified N007 as forgot_or_no_reminder and used other for the ambiguous N009.


In [10]:
clean_appointments = pd.read_csv("appointments_clean.csv")

merged = analysed.merge(
    clean_appointments[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left",
    validate="many_to_one",
)

assert len(merged) == 10
assert merged["reminder_sent"].notna().all()

display(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))
print("Notes linked to bookings without a reminder:",
      merged["reminder_sent"].eq("No").sum())

reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1


Notes linked to bookings without a reminder: 5


In [9]:
for note_id in ["N003", "N010"]:
    row_index = analysed.index[analysed["note_id"].eq(note_id)][0]  #Retry requests that returned no response
    note_text = notes.loc[notes["note_id"].eq(note_id), "note_text"].iloc[0]
    record = {"raw_response": ""}

    try:
        data = analyse_note(note_text, record=record)
        record.update({field: data.get(field) for field in result_fields})
        validate_insight(data)
        record["validation_status"] = "valid"
    except Exception as error:
        record["validation_status"] = "invalid"
        print(f"{note_id}: {type(error).__name__}: {error}")

    for column, value in record.items():
        analysed.at[row_index, column] = value

analysed.to_csv("notes_analysed.csv", index=False)
display(analysed[["note_id", "root_cause", "validation_status"]])
print("Total API calls:", api_calls)

,note_id,root_cause,validation_status
0,N001,forgot_or_no_reminder,valid
1,N002,distance_or_transport,valid
2,N003,cost_or_fee,valid
3,N004,other,skipped_blank
4,N005,illness_or_emergency,valid
5,N006,work_or_schedule,valid
6,N007,forgot_or_no_reminder,valid
7,N008,distance_or_transport,valid
8,N009,other,valid
9,N010,work_or_schedule,valid


Total API calls: 17
